In [1]:
import pandas as pd
import os

In [2]:
# 1. Load the raw dataset
# Adjust the path if your notebook is running from the root instead of the notebooks folder
data_path = '../data/raw/weather_prediction_dataset.csv'
df = pd.read_csv(data_path)

In [3]:
# 2. Pick a single city to start (the dataset contains 18 European locations)
# Let's use Basel, Switzerland as our test case
city_prefix = 'BASEL'
min_temp_col = f'{city_prefix}_temp_min'

In [4]:
# 3. Create the target continuous variable (Tomorrow's minimum temperature)
# We shift by -1 so that today's row contains tomorrow's value
df['target_future_min_temp'] = df[min_temp_col].shift(-1)

C:\Users\hp\AppData\Local\Temp\ipykernel_18324\1376384830.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['target_future_min_temp'] = df[min_temp_col].shift(-1)


In [5]:
# 4. Apply the threshold rule (<= 3.0°C) to generate the binary label
threshold = 3.0
df['frost_label'] = (df['target_future_min_temp'] <= threshold).astype(int)

C:\Users\hp\AppData\Local\Temp\ipykernel_18324\1231284101.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['frost_label'] = (df['target_future_min_temp'] <= threshold).astype(int)


In [6]:
# 5. Drop the very last row since we don't have tomorrow's data for it
df = df.dropna(subset=['target_future_min_temp'])

In [7]:
# 6. Check the class balance!
total_days = len(df)
frost_days = df['frost_label'].sum()
non_frost_days = total_days - frost_days

print(f"Total days analyzed for {city_prefix}: {total_days}")
print(f"Frost/Cold-Risk events (label=1): {frost_days}")
print(f"Safe days (label=0): {non_frost_days}")
print(f"Class imbalance: {(frost_days / total_days) * 100:.2f}% frost events")

Total days analyzed for BASEL: 3653
Frost/Cold-Risk events (label=1): 1083
Safe days (label=0): 2570
Class imbalance: 29.65% frost events


In [8]:
# 7. Save this interim dataset
# df.to_csv('../data/interim/basel_labeled.csv', index=False)